# n-shot prompting: helpdesk enquiry routing policy

Using n-shot prompting to teach a small model a new support-routing policy.

Compare **0, 1, 5 and 100 labelled examples** on the same **20 requests**.

### 1. Start the model server


In [1]:
%pip install -q pandas==2.2.3 ollama==0.6.2

from pathlib import Path
import sys
import pandas as pd

DEMOS = next(p / "demos" for p in (Path.cwd(), *Path.cwd().parents, Path.cwd() / "slides")
             if (p / "demos" / "notebook_setup.py").exists())
sys.path.insert(0, str(DEMOS))
from notebook_setup import local_client

MODEL = "gemma3:4b"
client = local_client(MODEL)



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Starting local Ollama...
Local model: gemma3:4b | digest: a2af6cc3eb7fa8be8504abaf9b04e88f17a119ec3f04a3addf55f92841195f5a


### 2. Our fictional company policy

We want to use an LLM to route customer enquirues to the right department:
| Request | Queue |
|---|---|
| Billing, payments or refunds | K2 |
| Sign-in or account permissions | K4 |
| Software faults | K1 |
| Any Partner-plan request, whatever the topic | K3 |

In [2]:
data = pd.read_csv(DEMOS / "3.1-model-adaptation/support-routing-demo.csv")
examples = data.iloc[:100]
test = data.iloc[100:].reset_index(drop=True)
examples.head()


,text,label
0,Standard plan. I was charged twice for this mo...,K2
1,Standard plan. I forgot my password and cannot...,K4
2,Standard plan. The app crashes whenever I expo...,K1
3,Partner plan. I was charged for the same order...,K3
4,Partner plan. The app closes as soon as I open...,K3


### 3. n-shot prompt builder

0-shot: the model knows the possible codes but not our company's rule or mapping.
1-shot: the model sees 1 correct example
...

In [3]:
def prompt(text, n):
    instructions = "Route the final support request using the company's routing policy.\n"
    instructions += "Reply with only one queue code: K1, K2, K3, or K4.\n\n"
    for example in examples.head(n).itertuples(index=False):
        instructions += f"Request: {example.text}\nQueue: {example.label}\n\n"
    return instructions + f"Request: {text}\nQueue:"

print(prompt(test.iloc[0].text, n=1))


Route the final support request using the company's routing policy.
Reply with only one queue code: K1, K2, K3, or K4.

Request: Standard plan. I was charged twice for this month's subscription.
Queue: K2

Request: Standard plan. I need the card charge for an accidental upgrade reversed.
Queue:


### 4. Compare 0, 1, 5 and 100 examples
The 8,192-token context leaves room for all 100 examples.


In [4]:
results = test[["text", "label"]].rename(columns={"label": "actual"}).copy()
for n in (0, 1, 5, 100):
    results[f"{n}-shot"] = [
        client.generate(
            model=MODEL, prompt=prompt(text, n),
            options={"temperature": 0, "num_predict": 8, "num_ctx": 8192},
        ).response.strip().upper()
        for text in test["text"]
    ]
    correct = (results[f"{n}-shot"] == results["actual"]).sum()
    print(f"{n:3}-shot: {correct}/20 correct")

results


  0-shot: 5/20 correct
  1-shot: 15/20 correct
  5-shot: 12/20 correct
100-shot: 16/20 correct


,text,actual,0-shot,1-shot,5-shot,100-shot
0,Standard plan. I need the card charge for an a...,K2,K1,K2,K1,K2
1,Partner plan. The dashboard crashes when we ch...,K3,K1,K3,K3,K3
2,Standard plan. The organisation invite says I ...,K4,K1,K1,K1,K4
3,Standard plan. The CSV download repeats the he...,K1,K1,K1,K1,K1
4,Standard plan. Our latest bill includes a lice...,K2,K1,K2,K1,K2
5,Standard plan. I replaced my handset and lost ...,K4,K1,K1,K4,K4
6,Partner plan. Our receipt shows the wrong lega...,K3,K1,K3,K3,K3
7,Standard plan. Clicking next in the setup wiza...,K1,K1,K1,K1,K1
8,Standard plan. A deleted task returns every ti...,K1,K1,K1,K1,K1
9,Standard plan. Please issue proof of payment f...,K2,K1,K2,K1,K2
